# Principal Component Analysis (PCA), Explained Simply
Squeezing many columns into a few, while keeping most of the information. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A supermarket has 200 breakfast cereals with six nutrition facts each (per 30 g serving), and a shelf type: kids, health or classic.

### 200 cereals, 6 numbers each
- `cereals.csv`: 200 cereals: sugar, fat, fibre and protein in grams, calories, salt in mg, and type
- `nutrition`: the six number columns we'll compress

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "pca/cereals.csv")
df = pd.read_csv(url)
nutrition = ["sugar_g", "fat_g", "fibre_g", "protein_g", "calories", "salt_mg"]
print(df.shape)
df.head()

### Which columns move together?
- `.corr()`: the correlation between every pair of columns: 1 = move together, -1 = opposite, 0 = unrelated
- `sugar and calories`: strongly linked: sweet cereals are higher in calories
- `salt_mg`: linked to nothing else

In [ ]:
print(df[nutrition].corr().round(2))

## 2. PCA
PCA finds new directions, called **principal components**. The first points where the data is most spread out; each next one is at right angles to the earlier ones and takes the most of what's left. Scale first, so no column wins just because of its units.

### Fit PCA
- `StandardScaler`: every column to mean 0, standard deviation 1
- `PCA()`: find all six components
- `explained_variance_ratio_`: the share of the total spread (variance) each component captures

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

Z = StandardScaler().fit_transform(df[nutrition])
pca = PCA().fit(Z)
print("share of the spread per component:")
print(pca.explained_variance_ratio_.round(3))

### How many components do we need?
- `np.cumsum`: a running total
- `the chart`: two components keep about three quarters, three keep over 90%

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

cum = np.cumsum(pca.explained_variance_ratio_)
plt.bar(range(1, 7), pca.explained_variance_ratio_, label="each component")
plt.plot(range(1, 7), cum, marker="o", color="red", label="running total")
plt.xlabel("component"); plt.ylabel("share of the spread")
plt.legend(); plt.show()
print("running total:", cum.round(2))

### What does each component mean?
- `pca.components_`: each component's recipe: how much of each column it mixes in
- `PC1`: plus sugar and calories, minus fibre and protein: 'sweet versus wholesome'
- `PC2`: mostly fat, plus calories and protein: 'rich'
- `PC3`: almost all salt

In [ ]:
loadings = pd.DataFrame(pca.components_[:3].T, index=nutrition,
                        columns=["PC1", "PC2", "PC3"]).round(2) + 0.0
print(loadings)

## 3. Six columns down to two
Each cereal becomes just two numbers: its position along PC1 and PC2.

### A map of all 200 cereals
- `PCA(n_components=2)`: keep just the first two components
- `fit_transform(Z)`: each cereal's two new numbers
- `the picture`: the three shelf types mostly separate, though PCA never saw the type column

In [ ]:
two = PCA(n_components=2).fit_transform(Z)
colors = {"kids": "tab:red", "health": "tab:green", "classic": "tab:blue"}
for t, part in df.groupby("type"):
    idx = part.index
    plt.scatter(two[idx, 0], two[idx, 1], c=colors[t], label=t, s=15)
plt.xlabel("PC1: wholesome  <->  sweet"); plt.ylabel("PC2: rich")
plt.legend(); plt.show()

## 4. Mini project: find similar cereals
Find the cereals closest to a given one on the PCA map.

### Cereals like Cereal 3
- `two[2]`: Cereal 3's position on the map (Python counts from 0)
- `dist`: the distance from every cereal to it
- `np.argsort(dist)[1:4]`: the three closest, skipping itself

In [ ]:
target = two[2]
dist = np.sqrt(((two - target) ** 2).sum(axis=1))
nearest = np.argsort(dist)[1:4]
cols = ["cereal", "type", "sugar_g", "fibre_g", "calories"]
print(df.loc[[2] + list(nearest), cols])

## Try it yourself
1. Run PCA without scaling. Which column takes over the first component, and why?
2. Plot component 1 against component 3. What does component 3 seem to measure?
3. Use PCA(n_components=0.95). How many components does it keep?